# Modeling & Evaluation — Todas as famílias de modelos

**Projeto:** Previsão do vencedor de confrontos profissionais de CS2
**Dataset:** [Counter-Strike Pro Matches (Kaggle, ektarr)](https://www.kaggle.com/datasets/ektarr/counter-strike-pro-matches)

Este notebook junta, num só, os 7 notebooks de modelação (um por família). Os dados, a validação e as métricas são iguais aos dos notebooks originais, por isso os resultados são comparáveis.

| # | Família | Modelo(s) |
|---|---|---|
| 01 | **Linear** | Regressão Logística |
| 02 | **Baseada em distância** | KNN |
| 03 | **Margem máxima** | SVM (RBF) |
| 04 | **Árvore única** | Decision Tree |
| 05 | **Bagging de árvores** | Random Forest, Extra Trees |
| 06 | **Boosting de árvores** | Gradient Boosting, HistGradientBoosting, XGBoost, LightGBM |
| 07 | **Rede neuronal** | MLP |

| | |
|---|---|
| **Alvo** | `target` = 1 se a equipa 1 vence o jogo/mapa, 0 caso contrário |
| **Entrada** | `cs2_matches_train.csv` e `cs2_matches_test.csv` em `MyDrive/CS2_Projeto/` (gerados no Data Preparation, split cronológico 80/20, sem *leakage*) |
| **Features** | 12 pré-jogo: Elo (`elo_team1`, `elo_team2`, `elo_diff`), taxa de vitórias, forma recente e h2h |
| **Validação** | `TimeSeriesSplit` (5 folds) com `GridSearchCV`, métrica de seleção: accuracy |

O Elo é usado como feature de treino, tal como as restantes. Não há baseline Elo e, nas métricas, aparece apenas o desempenho dos modelos.

## Sobre as famílias

- **Linear:** combina as features com pesos e passa o resultado por uma função logística. Simples, rápido e muito interpretável; não capta interações nem efeitos não lineares sem construção manual.
- **Baseada em distância (KNN):** guarda os jogos de treino e, para um jogo novo, vota no resultado dos *k* jogos mais parecidos. Sensível à escala (usa `StandardScaler`) e a ruído.
- **Margem máxima (SVM):** procura a fronteira que separa as classes com a maior margem; com kernel RBF a fronteira pode ser curva. Sensível à escala, treino pesado em muitos dados e sem probabilidades por defeito (o ROC AUC usa a função de decisão).
- **Árvore única:** regras do tipo "se `elo_diff` > 30 e forma > 0.6 então equipa 1 vence". Muito explicável, mas instável e com tendência a overfitting (por isso limita-se profundidade e folhas).
- **Bagging (Random Forest, Extra Trees):** muitas árvores em paralelo sobre amostras diferentes, com votação final; reduz a variância. A Extra Trees escolhe os cortes ao acaso.
- **Boosting:** árvores pequenas em sequência, cada uma a corrigir os erros das anteriores. Costuma ser a família mais forte em dados tabulares; exige cuidado com overfitting.
- **Rede neuronal (MLP):** camadas de neurónios com ativações não lineares. Com 12 features e ~8 600 jogos de treino tende a não superar o boosting; usa `early_stopping`.

## 1. Setup

In [ ]:
import os
import re
import time
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
RANDOM_STATE = 42

## 2. Carregamento dos dados

No Colab, monta o Google Drive e lê os CSV de `MyDrive/CS2_Projeto/` (a mesma pasta onde o Data Preparation os guarda). Se não os encontrar lá, tenta a pasta atual da sessão. Fora do Colab, usa a pasta atual.

In [ ]:
# Pasta no Google Drive onde o Data Preparation guardou os CSV (tem de ser a mesma usada lá)
PASTA_DRIVE = '/content/drive/MyDrive/CS2_Projeto/'
FICHEIROS = ['cs2_matches_train.csv', 'cs2_matches_test.csv']

PASTA = '.'   # fora do Colab usa-se a pasta atual
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(PASTA_DRIVE, exist_ok=True)
    PASTA = PASTA_DRIVE
except ImportError:
    print("Fora do Colab: a usar a pasta atual.")

caminhos = {f: os.path.join(PASTA, f) for f in FICHEIROS}

if not all(os.path.exists(c) for c in caminhos.values()):
    if all(os.path.exists(f) for f in FICHEIROS):
        # ficheiros soltos na sessão (ex.: /content/) em vez do Drive
        caminhos = {f: f for f in FICHEIROS}
        print("Ficheiros não encontrados no Drive; a usar os da pasta atual da sessão.")
    else:
        em_falta = [c for c in caminhos.values() if not os.path.exists(c)]
        raise FileNotFoundError(
            f"Não encontrei {em_falta}. Corre a secção 10 do Data Preparation a guardar em {PASTA_DRIVE}")

train = pd.read_csv(caminhos['cs2_matches_train.csv'])
test = pd.read_csv(caminhos['cs2_matches_test.csv'])
print("Dados lidos de:", os.path.dirname(os.path.abspath(caminhos['cs2_matches_train.csv'])))

X_train, y_train = train.drop(columns=['target']), train['target']
X_test, y_test = test.drop(columns=['target']), test['target']

assert list(X_train.columns) == list(X_test.columns), "Treino e teste com colunas diferentes"
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "Existem valores em falta"

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 3. Modelos e grelhas de hiperparâmetros

Todos os modelos ficam num único dicionário `modelos`, no formato `nome: (família, estimador, grelha)`.

Os modelos sensíveis à escala (Regressão Logística, KNN, SVM, MLP) vão dentro de um `Pipeline` com `StandardScaler`, ajustado só com os dados de treino de cada fold. Os modelos de árvores não precisam de escala.

In [ ]:
def com_escala(modelo):
    return Pipeline([('scaler', StandardScaler()), ('clf', modelo)])

modelos = {}   # nome -> (família, estimador, grelha)

# --- 01. Linear ---
modelos['Regressão Logística'] = ('Linear',
    com_escala(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100], 'clf__penalty': ['l2']})

# --- 02. Baseada em distância ---
modelos['KNN'] = ('Baseada em distância',
    com_escala(KNeighborsClassifier()),
    {'clf__n_neighbors': [15, 31, 61, 101], 'clf__weights': ['uniform', 'distance']})

# --- 03. Margem máxima ---
modelos['SVM (RBF)'] = ('Margem máxima',
    com_escala(SVC(kernel='rbf', random_state=RANDOM_STATE)),
    {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.1]})

# --- 04. Árvore única ---
modelos['Decision Tree'] = ('Árvore única',
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    {'max_depth': [3, 4, 5, 8], 'min_samples_leaf': [20, 50, 100]})

# --- 05. Bagging de árvores ---
modelos['Random Forest'] = ('Bagging de árvores',
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {'n_estimators': [200, 400], 'max_depth': [5, 10, None], 'min_samples_leaf': [5, 20]})

modelos['Extra Trees'] = ('Bagging de árvores',
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {'n_estimators': [200, 400], 'max_depth': [5, 10, None], 'min_samples_leaf': [5, 20]})

# --- 06. Boosting de árvores ---
modelos['Gradient Boosting'] = ('Boosting de árvores',
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3]})

modelos['HistGradientBoosting'] = ('Boosting de árvores',
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    {'learning_rate': [0.03, 0.1], 'max_depth': [3, 6], 'min_samples_leaf': [20, 50]})

try:
    from xgboost import XGBClassifier
    modelos['XGBoost'] = ('Boosting de árvores',
        XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'max_depth': [2, 4]})
except ImportError:
    print("XGBoost não está instalado — ignorado.")

try:
    from lightgbm import LGBMClassifier
    modelos['LightGBM'] = ('Boosting de árvores',
        LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'num_leaves': [7, 15]})
except ImportError:
    print("LightGBM não está instalado — ignorado.")

# --- 07. Rede neuronal ---
modelos['MLP (rede neuronal)'] = ('Rede neuronal',
    com_escala(MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE)),
    {'clf__hidden_layer_sizes': [(16,), (32,), (64, 32)], 'clf__alpha': [1e-3, 1e-2, 1e-1]})

ORDEM_FAMILIAS = list(dict.fromkeys(f for f, _, _ in modelos.values()))
FAMILIA_DE = {nome: fam for nome, (fam, _, _) in modelos.items()}
print(f"{len(modelos)} modelos em {len(ORDEM_FAMILIAS)} famílias:")
for fam in ORDEM_FAMILIAS:
    print(f"  {fam:<22} {[n for n, f in FAMILIA_DE.items() if f == fam]}")

## 4. Treino com validação cruzada temporal

Cada fold treina nos jogos mais antigos e valida nos seguintes, para a seleção de hiperparâmetros respeitar a ordem cronológica. Esta é a célula mais demorada do notebook (Random Forest, Extra Trees e SVM são os modelos mais pesados).

In [ ]:
cv_temporal = TimeSeriesSplit(n_splits=5)

melhores = {}
registo_cv = []

for nome, (familia, estimador, grelha) in modelos.items():
    t0 = time.time()
    gs = GridSearchCV(estimador, grelha, cv=cv_temporal, scoring='accuracy', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    melhores[nome] = gs.best_estimator_
    registo_cv.append({
        'Família': familia,
        'Modelo': nome,
        'Accuracy CV (média)': gs.best_score_,
        'Melhores hiperparâmetros': gs.best_params_,
        'Tempo (s)': round(time.time() - t0, 1),
    })
    print(f"{familia:<22} {nome:<22} CV accuracy = {gs.best_score_:.4f}  ({time.time() - t0:.1f}s)")

df_cv = pd.DataFrame(registo_cv).sort_values('Accuracy CV (média)', ascending=False).reset_index(drop=True)
df_cv

## 5. Avaliação no conjunto de teste

Jogos mais recentes, nunca vistos no treino. Métricas: accuracy, precision, recall, F1 e ROC AUC.

In [ ]:
def pontuacao(modelo, X):
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    return modelo.decision_function(X)

linhas, previsoes = [], {}
for nome, modelo in melhores.items():
    y_pred = modelo.predict(X_test)
    previsoes[nome] = y_pred
    linhas.append({
        'Família': FAMILIA_DE[nome],
        'Modelo': nome,
        'Accuracy (treino)': accuracy_score(y_train, modelo.predict(X_train)),
        'Accuracy (teste)': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, pontuacao(modelo, X_test)),
    })

resultados = (pd.DataFrame(linhas)
              .sort_values('Accuracy (teste)', ascending=False)
              .reset_index(drop=True))
resultados['Gap treino-teste'] = resultados['Accuracy (treino)'] - resultados['Accuracy (teste)']
resultados.round(4)

### 5.1 Comparação entre todos os modelos

In [ ]:
cores_fam = dict(zip(ORDEM_FAMILIAS, plt.cm.tab10(np.linspace(0, 1, 10))[:len(ORDEM_FAMILIAS)]))

ordem = resultados.sort_values('Accuracy (teste)')
fig, ax = plt.subplots(figsize=(9, 1.5 + 0.5 * len(ordem)))
barras = ax.barh(ordem['Modelo'], ordem['Accuracy (teste)'], color=[cores_fam[f] for f in ordem['Família']])
ax.bar_label(barras, fmt='%.3f', padding=3)
ax.set_xlim(max(0, ordem['Accuracy (teste)'].min() - 0.05), min(1, ordem['Accuracy (teste)'].max() + 0.05))
ax.set_xlabel('Accuracy no conjunto de teste')
ax.set_title('Accuracy de teste por modelo (cor = família)')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=cores_fam[f]) for f in ORDEM_FAMILIAS],
          labels=ORDEM_FAMILIAS, loc='lower right', fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Melhor modelo de cada família

In [ ]:
melhor_por_familia = (resultados.loc[resultados.groupby('Família')['Accuracy (teste)'].idxmax()]
                      .set_index('Família').loc[ORDEM_FAMILIAS].reset_index())
melhor_por_familia.round(4)

### 5.3 Treino vs teste

In [ ]:
ordem = resultados.sort_values('Accuracy (teste)', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(ordem))
ax.bar(x - 0.2, ordem['Accuracy (treino)'], width=0.4, label='Treino', color='#9ecae9')
ax.bar(x + 0.2, ordem['Accuracy (teste)'], width=0.4, label='Teste', color='#4C78A8')
ax.set_xticks(x)
ax.set_xticklabels(ordem['Modelo'], rotation=30, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Treino vs teste (diferença grande = overfitting)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Validação cruzada vs teste

In [ ]:
comp = df_cv[['Modelo', 'Accuracy CV (média)']].merge(
    resultados[['Modelo', 'Accuracy (teste)']], on='Modelo').sort_values('Accuracy (teste)', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(comp))
ax.bar(x - 0.2, comp['Accuracy CV (média)'], width=0.4, label='CV temporal (média)', color='#a1d99b')
ax.bar(x + 0.2, comp['Accuracy (teste)'], width=0.4, label='Teste', color='#4C78A8')
ax.set_xticks(x)
ax.set_xticklabels(comp['Modelo'], rotation=30, ha='right')
ax.set_ylabel('Accuracy')
ax.set_ylim(max(0, comp[['Accuracy CV (média)', 'Accuracy (teste)']].min().min() - 0.05), None)
ax.set_title('Accuracy em validação cruzada vs teste')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Melhor modelo global

Matriz de confusão e importância das features (por permutação, no conjunto de teste).

In [ ]:
nome_melhor = resultados.loc[0, 'Modelo']
melhor = melhores[nome_melhor]
print(f"Melhor modelo no teste: {nome_melhor} (família: {FAMILIA_DE[nome_melhor]})")
print(f"Accuracy de teste: {resultados.loc[0, 'Accuracy (teste)']:.4f}")
print(f"Hiperparâmetros: {df_cv.set_index('Modelo').loc[nome_melhor, 'Melhores hiperparâmetros']}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, previsoes[nome_melhor]),
                       display_labels=['Equipa 2 vence', 'Equipa 1 vence']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de confusão — {nome_melhor}')

perm = permutation_importance(melhor, X_test, y_test, scoring='accuracy',
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.DataFrame({'Feature': X_test.columns,
                     'Queda de accuracy': perm.importances_mean,
                     'Desvio-padrão': perm.importances_std})
       .sort_values('Queda de accuracy', ascending=True))
axes[1].barh(imp['Feature'], imp['Queda de accuracy'], xerr=imp['Desvio-padrão'], color='#F58518')
axes[1].set_xlabel('Queda média de accuracy ao baralhar a feature')
axes[1].set_title('Importância das features (permutação, teste)')
axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.1 Matriz de confusão do melhor modelo de cada família

In [ ]:
n = len(melhor_por_familia)
cols = 4
linhas_fig = int(np.ceil(n / cols))
fig, axes = plt.subplots(linhas_fig, cols, figsize=(4.2 * cols, 4 * linhas_fig))
axes = np.atleast_1d(axes).ravel()
for ax, (_, r) in zip(axes, melhor_por_familia.iterrows()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, previsoes[r['Modelo']]),
                           display_labels=['Eq. 2', 'Eq. 1']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['Família']}\n{r['Modelo']} ({r['Accuracy (teste)']:.3f})", fontsize=9)
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Guardar resultados

Os resultados e os modelos ficam em `CS2_Projeto/resultados/` (no Drive, se estiver montado), por isso não se perdem quando a sessão termina. Guarda-se o melhor modelo de cada família, mais o melhor modelo global.

In [ ]:
PASTA_SAIDA = os.path.join(PASTA, 'resultados')
os.makedirs(PASTA_SAIDA, exist_ok=True)

def slug(texto):
    t = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', t.lower()).strip('_')

resultados.round(4).to_csv(os.path.join(PASTA_SAIDA, 'resultados_todos_modelos.csv'), index=False)
melhor_por_familia.round(4).to_csv(os.path.join(PASTA_SAIDA, 'resultados_melhor_por_familia.csv'), index=False)
df_cv.to_csv(os.path.join(PASTA_SAIDA, 'resultados_cv_todos_modelos.csv'), index=False)

for _, r in melhor_por_familia.iterrows():
    joblib.dump(melhores[r['Modelo']], os.path.join(PASTA_SAIDA, f"modelo_{slug(r['Família'])}.joblib"))
joblib.dump(melhor, os.path.join(PASTA_SAIDA, 'modelo_melhor_global.joblib'))

print("Ficheiros guardados em:", os.path.abspath(PASTA_SAIDA))
print("  resultados_todos_modelos.csv, resultados_melhor_por_familia.csv, resultados_cv_todos_modelos.csv")
print("  modelo_<familia>.joblib (um por família) e modelo_melhor_global.joblib")

## 8. Notas para o relatório

- Este notebook substitui os 7 notebooks por família: a secção 5.2 dá o melhor modelo de cada família e a 5.1 compara todos os modelos de uma vez.
- Com ~2 mil jogos de teste, a margem de erro da accuracy ronda os ±2 pontos percentuais; diferenças pequenas entre modelos e famílias podem ser ruído.
- A coluna `Gap treino-teste` indica overfitting quando é grande e positiva.
- O modelo prevê o vencedor de cada **mapa**; para o confronto completo (BO3/BO5) seria preciso agregar por `match_id`.
- Se o XGBoost ou o LightGBM não estiverem instalados, são ignorados e o notebook corre na mesma com os restantes modelos.